In [ ]:
# Fix SSL certificate trust using the certifi CA bundle (must run before other imports)
# Avoid disabling TLS verification globally; configure trusted CAs instead
import os
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

In [ ]:
# Imports and shared configuration for the non-LLM baseline WSD pipeline
import re
from data_loader import load_sense_repo_by_round
from config import (
    ANNOTATION_CHUNKS,
    DATA_DIR,
    OUTPUT_DIR,
    SENSE_ID_FIELD,
    SENSE_COUNT_FIELD,
    SENSE_LIST_FIELD,
    SENSE_AINOTES_FIELD,
    SENSE_ORIGIN,
    S_ID,
    S_DEFINITION,
    S_LEMMA,
    S_UPOS,
    L_LEMMA,
    L_UPOS,
    L_MWE_ID,
)
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter

# ── Switch to change baseline method ────────────────────────────────────────
# Supported values: "LESK" | "INVENTORY_FIRST"
BASELINE_METHOD = "LESK"

# Derive origin tag and human-readable scorer name from the switch
_METHOD_META = {
    "LESK":  ("baseline_lesk",  "score_lesk_overlap"),
    "INVENTORY_FIRST": ("baseline_inventory_first", "score_inventory_first"),
}
if BASELINE_METHOD not in _METHOD_META:
    raise ValueError(f"Unknown BASELINE_METHOD={BASELINE_METHOD!r}. Choose 'LESK' or 'INVENTORY_FIRST'.")

ORIGIN_WSD, _SCORER_NAME = _METHOD_META[BASELINE_METHOD]
print(f"Baseline method : {BASELINE_METHOD}")
print(f"Origin tag      : {ORIGIN_WSD}")
print(f"Scorer function : {_SCORER_NAME}")

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 1

In [ ]:
# 2) Load sense repository and select corpus chunk
senses_df = load_sense_repo_by_round(round_number=ROUND)
print(f"Loaded sense repo for round {ROUND}: {len(senses_df)} senses")

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

chunk_idx = 0  # Select chunk index to process
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

begin, end = chunk_begin, chunk_end

test = False
tb, te = 0, 100  # 0-based slice within selected chunk when test=True
if test:
    sentences = sentences[tb:te]
    begin, end = chunk_begin + tb, chunk_begin + te - 1

In [ ]:
# 3) Baseline helper functions (self-contained — no sentence-transformers required)
from typing import Any, Dict, List, Sequence, Tuple
from preprocessing import (
    get_mwe_filtered_senses,
    get_mwe_tokens,
    get_token_senses,
    mark_mwe,
    mark_token,
    token_is_content_word,
    token_is_not_in_mwe,
)

_WORD_RE = re.compile(r"[\w]+", re.UNICODE)


def _shorten(text: str, max_length: int = 120) -> str:
    """Return text trimmed to max_length with ellipsis."""
    text = text.replace("\n", " ").strip()
    if len(text) <= max_length:
        return text
    return text[:max_length - 1] + "\u2026"


def _candidate_text_from_sense(sense: Dict[str, Any]) -> str:
    """Choose the best textual representation of a sense (definition > lemma > id)."""
    for key in (S_DEFINITION, "definition", "gloss"):
        value = sense.get(key)
        if value:
            text = str(value).strip()
            if text:
                return text
    lemma = sense.get(S_LEMMA, "")
    if lemma:
        lemma_text = str(lemma).strip()
        if lemma_text:
            return lemma_text
    return str(sense.get(S_ID, "unspecified sense"))


def score_lesk_overlap(
    context_text: str,
    candidate_texts: Sequence[str],
) -> List[Tuple[int, int]]:
    """Score each candidate by word-overlap with the context (Lesk algorithm).

    Returns list of (candidate_index, overlap_count) preserving original order.
    Tie-breaking relies on stable sort (earlier candidate wins).
    """
    ctx_tokens = set(_WORD_RE.findall(context_text.lower()))
    scores: List[Tuple[int, int]] = []
    for idx, cand_text in enumerate(candidate_texts):
        cand_tokens = set(_WORD_RE.findall(cand_text.lower()))
        scores.append((idx, len(ctx_tokens & cand_tokens)))
    return scores


def score_inventory_first(
    context_text: str,   # noqa: ARG001  (unused; signature matches score_lesk_overlap)
    candidate_texts: Sequence[str],
) -> List[Tuple[int, int]]:
    """Trivially score all candidates 0; first candidate wins by stable sort."""
    return [(idx, 0) for idx in range(len(candidate_texts))]


_SCORER = score_lesk_overlap if BASELINE_METHOD == "LESK" else score_inventory_first
print(f"Active scorer: {_SCORER.__name__}")


def _format_baseline_note(
    scores_sorted: List[Tuple[int, int]],
    candidate_texts: Sequence[str],
    senses: Sequence[Dict[str, Any]],
    method: str,
    scorer_name: str,
    max_items: int = 3,
) -> str:
    """Compact top-k note string in the same style as simple_wsd."""
    if not scores_sorted:
        return f"{method.lower()}_wsd({scorer_name}): no scored candidates"
    parts: List[str] = []
    for index, score in scores_sorted[:max_items]:
        sense_id = str(senses[index].get(S_ID, ""))
        description = _shorten(candidate_texts[index])
        parts.append(f"{sense_id}:{score}::{description}")
    return f"{method.lower()}_wsd({scorer_name}) - " + " - ".join(parts)


def process_senses_with_baseline_wsd(
    sentences,
    senses_df,
    *,
    scorer,
    method: str,
    scorer_name: str,
    origin_model: str,
    n: int = 2000,
    progress: bool = True,
    notes_top_k: int = 3,
):
    """Annotate sentences with INVENTORY_FIRST or LESK baseline WSD (no embeddings/LLMs)."""
    origin_first = "FIRST"
    sentence_total = len(sentences)
    processed_count = 0

    for sentence in sentences:
        # ── MWEs ────────────────────────────────────────────────────────────
        for mwe in getattr(sentence, "mwes", []):
            senses_df_slice = get_mwe_filtered_senses(mwe, senses_df)
            if senses_df_slice is None or senses_df_slice.empty:
                mwe_tokens = get_mwe_tokens(sentence, mwe)
                for token in mwe_tokens:
                    token.add_layer(SENSE_ID_FIELD, f"NEW_SENSE[{n}]")
                    token.add_layer(SENSE_COUNT_FIELD, f"0[{n}]")
                    token.add_layer(SENSE_ORIGIN, "None")
                n += 1
                continue

            senses_records = senses_df_slice.to_dict(orient="records")
            if not senses_records:
                mwe_tokens = get_mwe_tokens(sentence, mwe)
                for token in mwe_tokens:
                    token.add_layer(SENSE_ID_FIELD, f"NEW_SENSE[{n}]")
                    token.add_layer(SENSE_COUNT_FIELD, f"0[{n}]")
                    token.add_layer(SENSE_ORIGIN, "None")
                n += 1
                continue

            candidate_texts = [_candidate_text_from_sense(s) for s in senses_records]
            context_text = mark_mwe(mwe, sentence)
            scores = scorer(context_text, candidate_texts)
            scores_sorted = sorted(scores, key=lambda item: item[1], reverse=True)

            if not scores_sorted:
                best_index = 0
                notes = f"{method.lower()}_wsd({scorer_name}): fallback to first sense"
                origin = origin_first
            else:
                best_index = scores_sorted[0][0]
                notes = _format_baseline_note(
                    scores_sorted, candidate_texts, senses_records, method, scorer_name, notes_top_k
                )
                origin = origin_model

            sense_ids = [
                str(r.get(S_ID, "")) for r in senses_records if r.get(S_ID) is not None
            ]
            senses_candidates = ";".join(sense_ids)
            mwe_tokens = get_mwe_tokens(sentence, mwe)
            selected_sense = senses_records[best_index] if senses_records else {}
            sense_id_value = str(selected_sense.get(S_ID, "NEW_SENSE")) if senses_records else "NEW_SENSE"
            sense_count_value = str(len(senses_records))
            for token in mwe_tokens:
                token.add_layer(SENSE_ID_FIELD, f"{sense_id_value}[{n}]")
                token.add_layer(SENSE_COUNT_FIELD, f"{sense_count_value}[{n}]")
                token.add_layer(SENSE_LIST_FIELD, f"{senses_candidates}[{n}]")
                token.add_layer(SENSE_AINOTES_FIELD, f"{notes}[{n}]")
                token.add_layer(SENSE_ORIGIN, origin)
            n += 1

        # ── Single tokens ────────────────────────────────────────────────────
        for token in sentence.tokens:
            if not (token_is_content_word(token) and token_is_not_in_mwe(token)):
                continue

            target_word = token.layers.get(L_LEMMA, "_")
            if target_word == "_" or not target_word:
                continue
            if target_word[0].isdigit():
                continue

            senses_df_slice = get_token_senses(token, sentence, senses_df)
            senses_records = senses_df_slice.to_dict(orient="records")
            if not senses_records:
                token.add_layer(SENSE_ID_FIELD, "NEW_SENSE")
                token.add_layer(SENSE_COUNT_FIELD, "0")
                token.add_layer(SENSE_ORIGIN, "None")
                continue

            candidate_texts = [_candidate_text_from_sense(s) for s in senses_records]
            context_text = mark_token(token, sentence)
            scores = scorer(context_text, candidate_texts)
            scores_sorted = sorted(scores, key=lambda item: item[1], reverse=True)

            if not scores_sorted:
                best_index = 0
                notes = f"{method.lower()}_wsd({scorer_name}): fallback to first sense"
                origin = origin_first
            else:
                best_index = scores_sorted[0][0]
                notes = _format_baseline_note(
                    scores_sorted, candidate_texts, senses_records, method, scorer_name, notes_top_k
                )
                origin = origin_model

            sense_ids = [
                str(r.get(S_ID, "")) for r in senses_records if r.get(S_ID) is not None
            ]
            senses_candidates = ";".join(sense_ids)
            selected_sense = senses_records[best_index]
            sense_id_value = str(selected_sense.get(S_ID, "NEW_SENSE"))
            token.add_layer(SENSE_ID_FIELD, sense_id_value)
            token.add_layer(SENSE_AINOTES_FIELD, notes)
            token.add_layer(SENSE_COUNT_FIELD, str(len(senses_records)))
            token.add_layer(SENSE_LIST_FIELD, senses_candidates)
            token.add_layer(SENSE_ORIGIN, origin)

        processed_count += 1
        if progress:
            print(f"\rProcessed {processed_count}/{sentence_total} sentences (baseline_{method.lower()}).", end="")

    return sentences

In [ ]:
# 4) Annotate sentences with baseline WSD
sentences = process_senses_with_baseline_wsd(
    sentences,
    senses_df,
    scorer=_SCORER,
    method=BASELINE_METHOD,
    scorer_name=_SCORER_NAME,
    origin_model=ORIGIN_WSD,
    progress=True,
)

In [ ]:
# 5) Persist annotated corpus
ROUND_SUFFIX = f"_round{ROUND}"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_WSD}{ROUND_SUFFIX}.tsv")

incept_writer = InceptionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_WSD}{ROUND_SUFFIX}.tsv")

# BaselineWSD_sense.ipynb

This notebook mirrors `SimpleWSD_sense.ipynb` but runs a **non-LLM baseline WSD** instead of sentence-transformer embeddings.

Select a baseline by setting `BASELINE_METHOD` in cell 2:

| Value | Description | `ORIGIN_WSD` |
|-------|-------------|---------------|
| `"INVENTORY_FIRST"` | Pick the first candidate sense returned by candidate filtering (inventory order). | `baseline_inventory_first` |
| `"LESK"` | Classic Lesk: pick the candidate whose definition has the most word-overlap with the marked sentence context. Tie-break by earliest candidate. | `baseline_lesk` |

Outputs are written to `output/` with the same filename pattern as other notebooks:
- `LexiSense_{begin:04d}_{end:04d}_{ORIGIN_WSD}_round{ROUND}.tsv`
- `LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_WSD}_round{ROUND}.tsv`

No sentence-transformers or LLM API keys are required.